# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q fairlearn pandas numpy scikit-learn matplotlib seaborn faker
    print('Packages installed.')

# Algorithmic Fairness Audit In-Depth

As social scientists, we are often concerned with inequality and bias. When Machine Learning models are used to make decisions (e.g., hiring, loans, policing), they can reproduce or even amplify existing social biases. 

In this notebook, we will:
1.  Generate a synthetic "Hiring" dataset with intentional gender bias.
2.  Train a standard Logistic Regression model.
3.  Audit the model using `fairlearn` and manual metrics.
4.  Explore mitigation strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from faker import Faker

try:
    from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, true_positive_rate
    from fairlearn.preprocessing import CorrelationRemover
except ImportError:
    print("Fairlearn not installed. Some cells may fail.")

np.random.seed(42)
fake = Faker()

print("Libraries loaded.")

## 1. Generating Biased Data

We simulate a hiring scenario. 
- **Features**: `Experience` (Years), `Education` (Score), `Gender` (Male/Female).
- **Bias**: We explicitly make 'Female' candidates less likely to be hired for the same qualifications, or we make 'Experience' correlated with Gender in a way that favors Males.

In [ ]:
n = 1000

# 1. Gender (0=Female, 1=Male)
gender = np.random.choice([0, 1], size=n, p=[0.5, 0.5])

# 2. Experience (Correlated with Gender slightly)
# Males have slightly more experience on average in this biased world
experience = np.random.normal(5, 2, n) + (1.5 * gender)
experience = np.clip(experience, 0, 20)

# 3. Education Score (0-100)
education = np.random.normal(70, 10, n)

# 4. Hiring Decision (Target)
# Logic: Base score + Gender Bias + Randomness
# Score = 2*Experience + 0.5*Education
# Bias: Males get a +5 point bonus in the "decision logic" (simulating biased recruiters)
score = (2 * experience) + (0.5 * education) + (5 * gender) + np.random.normal(0, 5, n)

# Threshold for hiring
threshold = np.percentile(score, 70) # Top 30% get hired
hired = (score > threshold).astype(int)

df = pd.DataFrame({
    'Gender': gender,
    'Experience': experience,
    'Education': education,
    'Hired': hired
})

df['Gender_Str'] = df['Gender'].map({0: 'Female', 1: 'Male'})
df.head()

### Initial Check for Disparities
Let's check the base hiring rate by group.

In [ ]:
sns.barplot(data=df, x='Gender_Str', y='Hired')
plt.title("Hiring Rate by Gender")
plt.ylabel("Proportion Hired")
plt.show()

print(df.groupby('Gender_Str')['Hired'].mean())

## 2. Train a Model

We train a Logistic Regression to predict hiring. We include Gender as a feature (to show explicit bias) or exclude it (to show proxy bias via Experience). Let's exclude `Gender` explicitly but keep `Experience`.

In [ ]:
X = df[['Experience', 'Education']]
y = df['Hired']
A = df['Gender_Str'] # Sensitive Attribute

X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(X, y, A, test_size=0.3, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

## 3. Auditing the Model

We verify if the model treats groups differently. We use `fairlearn` to calculate metrics by group.

In [ ]:
metrics = {
    'selection_rate': selection_rate, # How many are hired?
    'false_positive_rate': false_positive_rate, # How many unqualified people got hired?
    'true_positive_rate': true_positive_rate, # How many qualified people got hired? (Recall)
}

metric_frame = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_test
)

print(metric_frame.by_group)

In [ ]:
metric_frame.by_group.plot(kind='bar', subplots=True, layout=[1, 3], figsize=(15, 5))
plt.show()

### Disparate Impact Ratio
A common legal standard (80% rule) is the Disparate Impact Ratio:
$$ \frac{P(\text{Hired} | \text{Unprivileged})}{P(\text{Hired} | \text{Privileged})} $$

If this is < 0.8, there is evidence of adverse impact.

In [ ]:
sr = metric_frame.by_group['selection_rate']
di_ratio = sr['Female'] / sr['Male']
print(f"Disparate Impact Ratio: {di_ratio:.2f}")

if di_ratio < 0.8:
    print("Warning: Potential Adverse Impact detected!")
else:
    print("Passes 80% rule.")

## 4. Conclusion

Even though we removed `Gender` from the training data, the model still showed bias because `Experience` was correlated with `Gender` in our biased historical data. This is known as **Proxy Bias**.

To fix this, we would need to:
1.  Collect better data.
2.  Use fairness-aware algorithms (e.g., `CorrelationRemover` or constraints in optimization).
3.  Adjust the decision threshold for different groups (though this can be legally complex).